#  Notebook 02: Ingestion & Vector Indexing Pipeline

Trong notebook này, chúng ta sẽ đi sâu vào từng giai đoạn của tầng Ingestion:
1. **Document Loaders**: Trích xuất dữ liệu từ PDF (`pypdf`) và DOCX (`python-docx`)
2. **Text Preprocessing**: Chuẩn hóa Unicode dựng sẵn (NFC) và làm sạch text tiếng Việt
3. **Recursive Chunking**: Cắt nhỏ văn bản theo cấu trúc ngữ pháp và đoạn văn bản tiếng Việt
4. **Vector Embedding**: Khám phá mô hình đa ngữ `BAAI/bge-m3`
5. **Qdrant Indexing**: Đẩy vectors kèm payload vào Qdrant Vector Database

## 1. Thiết lập Môi trường & Imports

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import get_config
from src.ingestion.loaders.pdf_loader import PDFLoader
from src.ingestion.loaders.docx_loader import DocxLoader
from src.utils.text import clean_text, normalize_vietnamese_unicode, count_words
from src.ingestion.chunkers.recursive_chunker import RecursiveCharacterChunker
from src.ingestion.embedder import BGEEmbedder
from src.ingestion.indexer import QdrantIndexer

cfg = get_config()
print(f"✅ Đã nạp thành công các module Ingestion. Data Raw: {cfg.data.raw_path}")

## 2. Thử nghiệm Document Loaders (PDF & DOCX)

Chúng ta sẽ thử đọc 2 file tài liệu thực tế có trong thư mục `data/raw/`.

In [ ]:
# 1. Đọc file PDF
pdf_file = cfg.data.raw_path / "paper_data mining.pdf"
if pdf_file.exists():
    pdf_loader = PDFLoader()
    pdf_docs = pdf_loader.load(pdf_file)
    print("=" * 65)
    print(f"📄 PDF: {pdf_file.name}")
    print(f"   - Số trang trích xuất được: {len(pdf_docs)}")
    print(f"   - Trang 1 metadata        : {pdf_docs[0].metadata}")
    print(f"   - Đoạn đầu trang 1        :\n{pdf_docs[0].text[:200]}...")

# 2. Đọc file DOCX
docx_file = cfg.data.raw_path / "paper_Phân tích và xử lí ảnh.docx"
if docx_file.exists():
    docx_loader = DocxLoader()
    docx_docs = docx_loader.load(docx_file)
    print("=" * 65)
    print(f"📝 DOCX: {docx_file.name}")
    print(f"   - Số phân đoạn (Headings/Tables): {len(docx_docs)}")
    print(f"   - Mục đầu tiên metadata         : {docx_docs[0].metadata}")
    print(f"   - Nội dung mục đầu tiên         :\n{docx_docs[0].text[:200]}...")
    print("=" * 65)

## 3. Tiền xử lý & Làm sạch Văn bản Tiếng Việt

Văn bản tiếng Việt từ PDF/Word thường gặp các lỗi:
- Bảng mã phân rã (NFD) dẫn tới tìm kiếm sai ký tự dấu
- Từ bị ngắt dòng có dấu gạch ngang (`phát-\ntriển` $\rightarrow$ `phát triển`)
- Khoảng trắng và dòng trống liên tiếp

In [ ]:
raw_sample = """
Trường   Đại  học    Sài Gòn.



Nghiên cứu áp dụng kỹ thuật phân đoạn ảnh vào việc chẩn-\nđoán bệnh lý nha khoa.
"""

cleaned_sample = clean_text(raw_sample)

print("=== VĂN BẢN GỐC ===")
print(repr(raw_sample))
print("\n=== VĂN BẢN SAU KHI LÀM SẠCH ===")
print(cleaned_sample)
print(f"\nSố từ: {count_words(cleaned_sample)}")

## 4. Cắt nhỏ Văn bản với Recursive Character Chunker

Cắt theo thứ tự ưu tiên: Đoạn văn (`\n\n`) $\rightarrow$ Dòng (`\n`) $\rightarrow$ Câu chấm (`. `), hỏi (`? `), cảm thán (`! `) $\rightarrow$ Dấu phẩy (`, `) $\rightarrow$ Từ (` `).

In [ ]:
chunker = RecursiveCharacterChunker(
    chunk_size=cfg.chunking.recursive.chunk_size,
    chunk_overlap=cfg.chunking.recursive.chunk_overlap,
)

# Lấy thử 3 documents từ DOCX để test chunk
sample_docs = docx_docs[:5]
sample_chunks = chunker.split_documents(sample_docs)

print("=" * 65)
print(f"📊 Cắt {len(sample_docs)} Documents thành {len(sample_chunks)} Chunks.")
print("-" * 65)

# Kiểm tra 1 chunk mẫu
c0 = sample_chunks[0]
print(f"🆔 Chunk ID   : {c0.chunk_id}")
print(f"📄 File       : {c0.metadata.get('file_name')} (Mục: '{c0.metadata.get('section_title')}')")
print(f"📏 Kích thước : {c0.char_count} ký tự ({c0.word_count} từ)")
print(f"📝 Nội dung   :\n{c0.text}")
print("=" * 65)

## 5. Thử nghiệm Tạo Vector Embedding với `BAAI/bge-m3`

Mô hình `BAAI/bge-m3` hỗ trợ tạo vector embedding 1024 chiều, cực tốt cho tiếng Việt.

In [ ]:
embedder = BGEEmbedder()

sample_sentence = "Mô hình học sâu DenseNet-201 phân loại bệnh về răng miệng đạt độ chính xác 93%."
vector = embedder.embed_query(sample_sentence)

print("=" * 65)
print(f" Embedding Model  : {embedder.model_name}")
print(f" Thiết bị (device): {embedder.device}")
print(f" Kích thước vector: {len(vector)} chiều (dimension)")
print(f" Vector 5 chiều đầu: {[round(x, 4) for x in vector[:5]]}")

# Kiểm tra chuẩn hóa L2 norm (phải xấp xỉ 1.0)
l2_norm = sum(x**2 for x in vector) ** 0.5
print(f" L2 Norm (chuẩn hóa Cosine): {l2_norm:.4f}")
print("=" * 65)

## 6. Kiểm tra Dữ liệu trên Qdrant Vector Store

Xem thông tin collection và các vectors đã được index.

In [ ]:
indexer = QdrantIndexer()

if indexer.is_connected():
    info = indexer.get_collection_info()
    print("=" * 65)
    print(f"✅ KẾT NỐI QDRANT COLLECTION THÀNH CÔNG:")
    print(f"   - Tên collection  : {info.get('name')}")
    print(f"   - Trạng thái      : {info.get('status')}")
    print(f"   - Tổng số vectors : {info.get('vectors_count')}")
    print(f"   - Kích thước      : {info.get('vector_size')}")
    print(f"   - Distance metric : {info.get('distance')}")
    print(f" Truy cập Dashboard tại: http://localhost:6333/dashboard")
    print("=" * 65)
else:
    print(" Không thể kết nối tới Qdrant. Hãy kiểm tra lệnh: docker compose up -d")